In [17]:
import os
os.environ['AWS_PROFILE'] = 'admin'
os.environ['HAVEN_DATABASE'] = 'haven'

import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import h3
from tqdm import tqdm

from mirrorverse.utils import read_data_w_cache
from mirrorverse.plotting import build_geojson, get_coords

from multiprocessing import Pool

import geopy.distance

In [2]:
sql = '''
select 
    date,
    h3_index,
    velocity_north,
    velocity_east
from 
    copernicus_physics
where 
    h3_resolution = 4
    and depth_bin = 25.0
    and extract(year from time) in (2015, 2016)
    and extract(day from time) = 1
'''
data = read_data_w_cache(sql)
print(data.shape)
data.head()

(416496, 4)


,date,h3_index,velocity_north,velocity_east
0,2015-12-01,8447003ffffffff,0.056783,-0.014412
1,2015-12-01,8447005ffffffff,0.125473,0.072681
2,2015-12-01,8447007ffffffff,0.219522,0.087326
3,2015-12-01,8447009ffffffff,0.306633,-0.033390
4,2015-12-01,844700bffffffff,0.109487,-0.057121


In [22]:
def get_radius(h3_index):
    coords = get_coords(h3_index)
    pairs = []
    for point in coords:
        pair = -float('inf')
        for counter_point in coords:
            distance = geopy.distance.geodesic(point[::-1], counter_point[::-1]).km
            if distance > pair:
                pair = distance
        pairs.append(distance)
    return np.mean(pairs)

h3_indices = data[['h3_index']].drop_duplicates()

with Pool(8) as p:
    radii = p.map(get_radius, h3_indices['h3_index'])

h3_indices['radius'] = np.array(radii)

data = data[[c for c in data.columns if c != 'radius']].merge(h3_indices)
print(data.shape)
data.head()

(416496, 5)


,date,h3_index,velocity_north,velocity_east,radius
0,2015-12-01,8447003ffffffff,0.056783,-0.014412,28.710980
1,2015-12-01,8447005ffffffff,0.125473,0.072681,28.856639
2,2015-12-01,8447007ffffffff,0.219522,0.087326,28.768891
3,2015-12-01,8447009ffffffff,0.306633,-0.033390,28.827272
4,2015-12-01,844700bffffffff,0.109487,-0.057121,28.740239


In [24]:
data['speed'] = data.apply(lambda r: (r['velocity_north'] ** 2 + r['velocity_east'] ** 2) ** 0.5 / 1000 * 3600 * 24, axis=1)
data.head()

,date,h3_index,velocity_north,velocity_east,radius,speed
0,2015-12-01,8447003ffffffff,0.056783,-0.014412,28.710980,5.061605
1,2015-12-01,8447005ffffffff,0.125473,0.072681,28.856639,12.528323
2,2015-12-01,8447007ffffffff,0.219522,0.087326,28.768891,20.412314
3,2015-12-01,8447009ffffffff,0.306633,-0.033390,28.827272,26.649707
4,2015-12-01,844700bffffffff,0.109487,-0.057121,28.740239,10.669718


In [25]:
data['speed'].describe()

count    416496.000000
mean         11.184766
std          12.579442
min           0.000000
25%           4.292392
50%           7.721077
75%          13.164847
max         166.053717
Name: speed, dtype: float64

In [26]:
7 / 24

0.2916666666666667